In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 00 Create Catalog and Schemas
# MAGIC
# MAGIC Creates the main catalog and schemas used by the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Layer
# MAGIC Setup
# MAGIC
# MAGIC ## Architecture
# MAGIC The project uses a simplified Medallion Architecture:
# MAGIC
# MAGIC Bronze → Silver → Gold → Marts
# MAGIC
# MAGIC ## Outputs
# MAGIC
# MAGIC Catalog:
# MAGIC - `brazil_legislative_analytics`
# MAGIC
# MAGIC Schemas:
# MAGIC - `audit`
# MAGIC - `bronze`
# MAGIC - `silver`
# MAGIC - `gold`
# MAGIC - `marts`
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python variables are written in English.
# MAGIC - Schema names follow technical Medallion standards.
# MAGIC - Schema comments are written in English.
# MAGIC - Schema comments support governance, traceability and technical documentation.

# COMMAND ----------

from datetime import datetime

# COMMAND ----------

CATALOG_NAME = "brazil_legislative_analytics"

REQUIRED_SCHEMAS = {
    "audit": "Governance schema containing audit logs, error logs and data quality validation logs.",
    "bronze": "Raw ingestion layer containing data extracted from source APIs or CSV fallback files with minimal transformation.",
    "silver": "Unified curation layer responsible for cleansing, typing, standardization, deduplication, validation and enrichment.",
    "gold": "Dimensional analytical layer containing dimensions and fact tables following the Star Schema model.",
    "marts": "Analytical consumption layer containing business-oriented marts and final analytical outputs.",
}

# COMMAND ----------

execution_timestamp = datetime.now()

print("=" * 80)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("00 - CREATE CATALOG AND SCHEMAS")
print("=" * 80)
print(f"Execution timestamp: {execution_timestamp}")
print(f"Target catalog: {CATALOG_NAME}")
print("=" * 80)

# COMMAND ----------

# ============================================================
# CREATE PROJECT CATALOG
# ============================================================

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}
""")

print(f"Catalog validated: {CATALOG_NAME}")

# COMMAND ----------

# ============================================================
# CREATE MEDALLION AND GOVERNANCE SCHEMAS
# ============================================================

for schema_name, schema_comment in REQUIRED_SCHEMAS.items():
    full_schema_name = f"{CATALOG_NAME}.{schema_name}"

    spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {full_schema_name}
    """)

    spark.sql(f"""
    COMMENT ON SCHEMA {full_schema_name}
    IS '{schema_comment}'
    """)

    print(f"Schema validated: {full_schema_name}")

# COMMAND ----------

# ============================================================
# SET ACTIVE CATALOG
# ============================================================
#
# The active catalog is explicitly set to avoid accidental
# table creation in the default workspace catalog.
#
# ============================================================

spark.sql(f"USE CATALOG {CATALOG_NAME}")

print(f"Active catalog set to: {CATALOG_NAME}")

# COMMAND ----------

# ============================================================
# VALIDATE CREATED SCHEMAS
# ============================================================

schemas_df = spark.sql(f"""
SHOW SCHEMAS IN {CATALOG_NAME}
""")

display(schemas_df)

# COMMAND ----------

# ============================================================
# SETUP VALIDATION SUMMARY
# ============================================================

existing_schemas = [
    row["databaseName"]
    for row in schemas_df.collect()
]

missing_schemas = [
    schema_name
    for schema_name in REQUIRED_SCHEMAS.keys()
    if schema_name not in existing_schemas
]

print("=" * 80)
print("SETUP VALIDATION SUMMARY")
print("=" * 80)
print(f"Catalog: {CATALOG_NAME}")

for schema_name in REQUIRED_SCHEMAS.keys():
    print(f"Schema: {schema_name}")

if missing_schemas:
    raise Exception(
        f"Catalog/schema setup validation failed. "
        f"Missing schemas: {missing_schemas}"
    )

print("=" * 80)
print("CATALOG AND SCHEMAS CREATED SUCCESSFULLY")
print("=" * 80)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Created Structures
# MAGIC
# MAGIC ### Catalog
# MAGIC - `brazil_legislative_analytics`
# MAGIC
# MAGIC ### Schemas
# MAGIC - `audit`
# MAGIC - `bronze`
# MAGIC - `silver`
# MAGIC - `gold`
# MAGIC - `marts`
# MAGIC
# MAGIC ## Governance Notes
# MAGIC
# MAGIC - All schemas include comments for documentation and governance.
# MAGIC - The active catalog is explicitly set to avoid accidental table creation in the default schema.
# MAGIC - The Silver layer is unified and replaces the previous `silver_base` and `silver_curated` split.
# MAGIC - Future notebooks must always write tables using the fully qualified name:
# MAGIC
# MAGIC ```text
# MAGIC catalog.schema.table_name
# MAGIC ```
# MAGIC
# MAGIC Example:
# MAGIC
# MAGIC ```text
# MAGIC brazil_legislative_analytics.bronze.br_deputados
# MAGIC brazil_legislative_analytics.silver.slv_deputados
# MAGIC brazil_legislative_analytics.gold.dm_deputado
# MAGIC brazil_legislative_analytics.marts.am_atlas_frentes
# MAGIC ```
# MAGIC
# MAGIC ## Next Step
# MAGIC Execute:
# MAGIC
# MAGIC `01_project_config`

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
00 - CREATE CATALOG AND SCHEMAS
Execution timestamp: 2026-05-20 02:12:47.533489
Target catalog: brazil_legislative_analytics
Catalog validated: brazil_legislative_analytics
Schema validated: brazil_legislative_analytics.audit
Schema validated: brazil_legislative_analytics.bronze
Schema validated: brazil_legislative_analytics.silver
Schema validated: brazil_legislative_analytics.gold
Schema validated: brazil_legislative_analytics.marts
Active catalog set to: brazil_legislative_analytics


databaseName
audit
bronze
gold
information_schema
marts
silver


SETUP VALIDATION SUMMARY
Catalog: brazil_legislative_analytics
Schema: audit
Schema: bronze
Schema: silver
Schema: gold
Schema: marts
CATALOG AND SCHEMAS CREATED SUCCESSFULLY
